Francesca Conti - francesca.conti22@studio.unibo.it (Student ID: 111)

Matteo Preda - matteo.preda@studio.unibo.it (Student ID: 112)


# **Product Recognition of Books**

## Image Processing and Computer Vision - Assignment Module \#1

Contacts:

- Prof. Giuseppe Lisanti -> giuseppe.lisanti@unibo.it
- Prof. Samuele Salti -> samuele.salti@unibo.it
- Alex Costanzino -> alex.costanzino@unibo.it
- Francesco Ballerini -> francesco.ballerini4@unibo.it


Computer vision-based object detection techniques can be applied in library or bookstore settings to build a system that identifies books on shelves.

Such a system could assist in:

- Helping visually impaired users locate books by title/author;
- Automating inventory management (e.g., detecting misplaced or out-of-stock books);
- Enabling faster book retrieval by recognizing spine text or cover designs.


## Task

Develop a computer vision system that, given a reference image for each book, is able to identify such book from one picture of a shelf.

<figure>
<a href="https://ibb.co/pvLVjbM5"><img src="https://i.ibb.co/svVx9bNz/example.png" alt="example" border="0"></a>
</figure>

For each type of product displayed on the shelf, the system should compute a bounding box aligned with the book spine or cover and report:

1. Number of instances;
1. Dimension of each instance (area in pixel of the bounding box that encloses each one of them);
1. Position in the image reference system of each instance (four corners of the bounding box that enclose them);
1. Overlay of the bounding boxes on the scene images.

<font color="red"><b>Each step of this assignment must be solved using traditional computer vision techniques.</b></font>

#### Example of expected output

```
Book 0 - 2 instance(s) found:
  Instance 1 {top_left: (100,200), top_right: (110, 220), bottom_left: (10, 202), bottom_right: (10, 208), area: 230px}
  Instance 2 {top_left: (90,310), top_right: (95, 340), bottom_left: (24, 205), bottom_right: (23, 234), area: 205px}
Book 1 – 1 instance(s) found:
.
.
.
```


## Data

Two folders of images are provided:

- **Models**: contains one reference image for each product that the system should be able to identify;
- **Scenes**: contains different shelve pictures to test the developed algorithm in different scenarios.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/AssignmentsIPCV/dataset.zip ./
!unzip dataset.zip

Mounted at /content/drive
Archive:  dataset.zip
   creating: dataset/
   creating: dataset/scenes/
  inflating: dataset/.DS_Store       
  inflating: __MACOSX/dataset/._.DS_Store  
   creating: dataset/models/
  inflating: dataset/scenes/scene_9.jpg  
  inflating: __MACOSX/dataset/scenes/._scene_9.jpg  
  inflating: dataset/scenes/scene_8.jpg  
  inflating: __MACOSX/dataset/scenes/._scene_8.jpg  
  inflating: dataset/scenes/scene_20.jpg  
  inflating: __MACOSX/dataset/scenes/._scene_20.jpg  
  inflating: dataset/scenes/scene_21.jpg  
  inflating: __MACOSX/dataset/scenes/._scene_21.jpg  
  inflating: dataset/scenes/scene_23.jpg  
  inflating: __MACOSX/dataset/scenes/._scene_23.jpg  
  inflating: dataset/scenes/scene_22.jpg  
  inflating: __MACOSX/dataset/scenes/._scene_22.jpg  
  inflating: dataset/scenes/scene_26.jpg  
  inflating: __MACOSX/dataset/scenes/._scene_26.jpg  
  inflating: dataset/scenes/scene_27.jpg  
  inflating: __MACOSX/dataset/scenes/._scene_27.jpg  
  inflating: datas

In [15]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

# Set latex font for plots cmu
plt.rcParams["font.family"] = "cmr10"
plt.rcParams["font.size"] = 16

In [3]:
MODELS_FOLDER = "./dataset/models"
SCENES_FOLDER = "./dataset/scenes"

# read all the images inside all models and scene
models_images = []
scenes_images = []

for folder, image_list, image_name, image_ext in [
    (MODELS_FOLDER, models_images, "model", "png"),
    (SCENES_FOLDER, scenes_images, "scene", "jpg"),
]:
    num_images = len(
        [name for name in os.listdir(folder) if name.startswith(image_name)]
    )
    for i in range(num_images):
        image_list.append(
            cv2.imread(os.path.join(folder, f"{image_name}_{i}.{image_ext}"))
        )

# Methodology

The proposed solution is based on the extraction and matching of SIFT features between the model images and the scene images. The main steps of the algorithm are as follows:

1. Preprocess the images to enhance SIFT performance (Gaussian blur and histogram equalization). [**Discarded at the end because it did not improve performance**]
2. Extract SIFT keypoints and descriptors from both model and scene images (SIFT is a robust feature descriptor).
3. Match the descriptors using a $k$-NN matcher with $k=5$ (we did not use $k=2$ because a higher $k$ helps in scenes with multiple instances of the same model).
4. Apply Lowe's ratio test to filter out weak matches.
5. We estimate an affine transformation using RANSAC to robustly estimate the mapping between the model and scene keypoints. (**We did not use Homography estimation because the books are mostly planar objects and the perspective distortion is not significant in our dataset. In this way we can inject an inductive bias towards the expected book layout.**)
6. Use the estimated affine transformation to project the corners of the model image onto the scene image, obtaining the bounding boxes for each detected book instance.
7. Draw a black bounding box using the projected corners and repeat the process for each model image on the scene image. (**This is done to avoid detecting the same instance multiple times.**)


In [4]:
def instance_detect(
    scene_img,
    model_img,
    preprocess=None,
    plot_matches: bool = True,
    plot_rectangle: bool = True,
    plot_descriptors: bool = False,
    debug: bool = False,
):
    """
    Given a scene image and a model image, detect a single instance of the model in the scene.
    """

    # SIFT only works on a single channel (grayscale) images
    model_gray = cv2.cvtColor(model_img, cv2.COLOR_BGR2GRAY)
    scene_gray = cv2.cvtColor(scene_img, cv2.COLOR_BGR2GRAY)

    if preprocess is not None:
        model_gray = preprocess(model_gray)
        scene_gray = preprocess(scene_gray)

    # Define and compute SIFT keypoints and descriptors
    sift = cv2.SIFT_create(
        nfeatures=0,
        nOctaveLayers=7,  # Increased to capture more details
        contrastThreshold=0.02,  # Lowered to detect more keypoints
        edgeThreshold=10,
        sigma=1.2,
    )
    model_keypoints, des_model = sift.detectAndCompute(model_gray, None)
    scene_keypoints, des_scene = sift.detectAndCompute(scene_gray, None)

    if plot_descriptors:
        model_img_kp = cv2.drawKeypoints(
            model_img.copy(),
            model_keypoints,
            None,
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
        )
        scene_img_kp = cv2.drawKeypoints(
            scene_img.copy(),
            scene_keypoints,
            None,
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
        )

        plt.figure(figsize=(20, 10))
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(model_img_kp, cv2.COLOR_BGR2RGB))
        plt.title("Model Keypoints")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(cv2.cvtColor(scene_img_kp, cv2.COLOR_BGR2RGB))
        plt.title("Scene Keypoints")
        plt.axis("off")
        plt.show()

    # Match descriptors using Brute Force Matcher
    bf = cv2.BFMatcher(cv2.NORM_L2)
    knn_matches = bf.knnMatch(des_model, des_scene, k=5)

    # Filter out the good matches
    good_matches = []
    for matches in knn_matches:
        for i in range(len(matches) - 1):
            if matches[i].distance < 0.6 * matches[i + 1].distance:  # Lowe's ratio test
                good_matches.append(matches[i])

    if plot_matches:
        # Plot the distribution of distances
        distances = [m.distance for m in good_matches]
        plt.figure(figsize=(10, 5))
        plt.hist(distances, bins=30, color="blue", alpha=0.7)
        plt.title("Distribution of Descriptor Distances for Good Matches")
        plt.xlabel("Distance")
        plt.ylabel("Frequency")
        plt.grid()
        plt.show()

    if plot_matches:
        plt.figure(figsize=(20, 10))
        img_matches = cv2.drawMatches(
            model_img.copy(),
            model_keypoints,
            scene_img.copy(),
            scene_keypoints,
            good_matches,
            None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
        )
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
        plt.title("SIFT Matches")
        plt.show()

    model_matching_keypoints = np.float32(
        [model_keypoints[m.queryIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)
    scene_matching_keypoints = np.float32(
        [scene_keypoints[m.trainIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)

    # We empirically found that at least 5 matches are needed to compute a reliable homography
    if len(good_matches) < 5:
        if debug:
            print("Not enough matches found.")
        return [[[0, 0], [0, 0], [0, 0], [0, 0]]]

    # We do not use findHomography because we use the inductive bias that the object is planar and we can use
    # affine transformations only (no perspective distortion)
    M, inliers = cv2.estimateAffinePartial2D(
        model_matching_keypoints,
        scene_matching_keypoints,
        method=cv2.RANSAC,
        ransacReprojThreshold=5.0,
    )

    # Convert affine matrix to homography matrix
    H = np.vstack([M, [0, 0, 1]])

    inlier_mask = inliers.ravel().tolist()
    num_inliers = sum(inlier_mask)

    if debug:
        print(f"RANSAC inliers: {num_inliers} / {len(good_matches)}")

    # We empirically found that at least 1/3 of the matches should be inliers to consider a valid detection
    if num_inliers / len(good_matches) < 1 / 3 or num_inliers < 5:
        if debug:
            print("Not enough inliers after RANSAC.")
        return [[[0, 0], [0, 0], [0, 0], [0, 0]]]

    # Get the corners of the model image
    model_corners = np.float32(
        [
            [0, 0],
            [model_img.shape[1], 0],
            [model_img.shape[1], model_img.shape[0]],
            [0, model_img.shape[0]],
        ]
    ).reshape(-1, 1, 2)

    if H is None:
        if debug:
            print("Homography could not be computed.")
        return [[[0, 0], [0, 0], [0, 0], [0, 0]]]

    # Transform the corners to the scene image using the homography
    scene_corners = cv2.perspectiveTransform(model_corners, H)
    if plot_rectangle:
        scene_img_with_box = scene_img.copy()
        cv2.polylines(
            scene_img_with_box,
            [np.int32(scene_corners)],
            isClosed=True,
            color=(0, 255, 0),
            thickness=5,
        )

        plt.subplot(1, 2, 2)
        plt.imshow(cv2.cvtColor(scene_img_with_box, cv2.COLOR_BGR2RGB))
        plt.title("Detected Model")
        plt.axis("off")

    plt.show()
    return scene_corners


def is_rectangle_valid(rectangle, image_shape):
    # Criterias:
    # 1. Should have 4 points
    # 2. Area should be within a certain reasonable range
    # 3. Check if the shape is roughly rectangular
    # 4. Check if the rectangle is within the image bounds

    if len(rectangle) != 4:
        return False, "Not 4 points"

    # Check area
    area = cv2.contourArea(rectangle)
    if area < 1000 or area > 100000:
        return False, "Invalid area"

    # Check ratio of area to bounding box area (extent)
    x, y, w, h = cv2.boundingRect(rectangle)
    bounding_box_area = w * h
    rectangle_area = cv2.contourArea(rectangle)
    extent = rectangle_area / bounding_box_area

    if extent < 0.5:
        return False, "Not rectangular enough"

    # Check if the rectangle is within the image bounds
    for p in rectangle:
        x, y = p[0]
        if x < 0 or x >= image_shape[1] or y < 0 < 0 or y >= image_shape[0]:
            return False, "Point out of bounds"

    return True, "Valid rectangle"

## Plotting function


In [5]:
def plot_detections(scenes_images, matches, SCENE_IDX=0):
    """
    Plot the scene image with detected models outlined and the model images with colored borders.
    """
    import cv2
    import matplotlib.pyplot as plt
    import numpy as np

    TARGET_HEIGHT = 600

    COLORS = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (0, 255, 255)]

    # Resize scene image
    h, w = scenes_images[SCENE_IDX].shape[:2]
    scale = TARGET_HEIGHT / h
    scene_resized = cv2.resize(
        scenes_images[SCENE_IDX], (int(w * scale), TARGET_HEIGHT)
    )

    # Draw polylines on resized image
    for i, (model_image, scene_corners, idx, scene_idx) in enumerate(matches):
        # Scale corners to match resized image
        scaled_corners = scene_corners * scale
        scene_resized = cv2.polylines(
            scene_resized,
            [np.int32(scaled_corners)],
            isClosed=True,
            color=COLORS[i % len(COLORS)],
            thickness=5,
        )

    plt.figure(figsize=(5 + 2 * (len(matches)), 5))

    plt.subplot(1, len(matches) + 1, 1)
    plt.imshow(cv2.cvtColor(scene_resized, cv2.COLOR_BGR2RGB))
    plt.title(f"Scene {SCENE_IDX}")
    plt.axis("off")

    # Plot each model image
    for i, (model_image, scene_corners, idx, scene_idx) in enumerate(matches):
        h, w = model_image.shape[:2]
        scale = TARGET_HEIGHT / h
        model_resized = cv2.resize(model_image, (int(w * scale), TARGET_HEIGHT))

        # Draw colored border
        cv2.polylines(
            model_resized,
            [
                np.int32(
                    [
                        [0, 0],
                        [model_resized.shape[1], 0],
                        [model_resized.shape[1], model_resized.shape[0]],
                        [0, model_resized.shape[0]],
                    ]
                )
            ],
            isClosed=True,
            color=COLORS[i % len(COLORS)],
            thickness=20,
        )

        plt.subplot(1, len(matches) + 1, i + 2)
        plt.imshow(cv2.cvtColor(model_resized, cv2.COLOR_BGR2RGB))
        plt.title(f"Model {idx}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## Experimental Setup and Hyperparameter Optimization Attempts

### 1. Objective
The goal of this phase was to identify a robust set of hyperparameters for the SIFT-based detection pipeline and the subsequent geometric verification steps.  
We aimed to optimize detection quality and consistency across different scenes containing various book models.

---

### 2. Hyperparameters Explored
An initial **grid search** was conducted over a range of parameters influencing both feature extraction and geometric verification:

- **SIFT parameters:**
  - `nfeatures`
  - `nOctaveLayers`
  - `contrastThreshold`
  - `edgeThreshold`
  - `sigma`

- **Matching and filtering parameters:**
  - `k` (number of nearest neighbors for KNN matching)
  - `ratio_test` (Lowe’s ratio test threshold)
  - `ransac_threshold` (RANSAC reprojection error)
  - `min_matches` (minimum number of valid matches)

- **Preprocessing parameters:**
  - `blur_kernel` (Gaussian blur kernel size)
  - `use_clahe` (adaptive histogram equalization)
  - `use_hist_eq` (standard histogram equalization)

---

### 3. Ground Truth and Evaluation Metric
To quantitatively evaluate detection performance, we manually annotated each scene by drawing the **ground truth bounding polygons** corresponding to the known models.

For each scene: We computed the **ground truth area** covered by the manually annotated matches, then we compared it with the **total detected area** produced by our algorithm under each hyperparameter configuration.

The **optimization metric** was defined as the absolute difference between the total detected area and the ground truth area, complemented by:
- Mean absolute error (MAE)
- Standard deviation of the error
- Relative error with respect to the ground truth

---

### 4. Results and Decision
Although this approach provided an initial quantitative criterion, we ultimately **discarded the area-based optimization process**.

The reason is that the **total detected area per scene** carries no spatial information about *where* the matches occur.  
Two configurations could yield similar total areas while producing detections in completely different (and possibly incorrect) regions.

Consequently, while the area-based metric was useful for coarse tuning, it failed to ensure spatial accuracy of detections.  
We therefore opted for a qualitative validation approach combined with targeted parameter adjustments guided by visual inspection of match consistency and geometric stability.

---

### 5. Outcome
The final parameter configuration was selected empirically, balancing:
- The density and stability of SIFT keypoints,
- The robustness of RANSAC inliers,
- And the visual correctness of model localization within each scene.

This combination produced the most reliable results across different lighting conditions and scene compositions.



In [18]:
def visualize_ground_truth_annotations(json_file_path, scenes_images, scene_indices=None):
    """
    Visualize ground truth annotations from JSON file.

    Args:
        json_file_path: Path to labels.json file
        scenes_images: List of scene images
        scene_indices: List of scene indices to visualize (None = all)
    """
    # Load JSON
    with open(json_file_path, 'r') as f:
        labels = json.load(f)

    # Map scene_name -> scene_index
    scene_mapping = {}
    for scene_name in labels.keys():
        scene_idx = int(scene_name.split('_')[1].split('.')[0])
        scene_mapping[scene_idx] = scene_name

    # If not specified, visualize all scenes
    if scene_indices is None:
        scene_indices = sorted(scene_mapping.keys())

    for scene_idx in scene_indices:
        if scene_idx not in scene_mapping:
            print(f"Scene {scene_idx} not found in JSON")
            continue

        scene_name = scene_mapping[scene_idx]
        scene_data = labels[scene_name]

        # Get image
        if scene_idx >= len(scenes_images):
            print(f"Scene {scene_idx} not found in images")
            continue

        scene_image = scenes_images[scene_idx].copy()

        # Create figure
        fig, ax = plt.subplots(1, 1, figsize=(15, 10))

        # Show image
        ax.imshow(cv2.cvtColor(scene_image, cv2.COLOR_BGR2RGB))

        # Colors for different models
        colors = plt.cm.Set3(np.linspace(0, 1, 12))
        model_color_map = {}
        color_idx = 0

        # Draw each region
        for region_idx, region in enumerate(scene_data['regions']):
            points = np.array(region['points'])
            label = region['label']

            # Assign color to model
            if label not in model_color_map:
                model_color_map[label] = colors[color_idx % len(colors)]
                color_idx += 1

            color = model_color_map[label]

            # Draw polygon
            polygon = Polygon(points, fill=True, alpha=0.3,
                            facecolor=color, edgecolor=color, linewidth=3)
            ax.add_patch(polygon)

            # Add label to polygon center
            centroid = np.mean(points, axis=0)
            ax.text(centroid[0], centroid[1], f"{label}\n#{region_idx+1}",
                   color='white', fontsize=10, weight='bold',
                   ha='center', va='center',
                   bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))

            # Draw polygon points
            ax.plot(points[:, 0], points[:, 1], 'o', color=color, markersize=5)

        # Title with statistics
        total_books = len(scene_data['regions'])
        count_info = scene_data['count']
        title = f"Scene {scene_idx} - {total_books} total books\n"
        title += ", ".join([f"{model}: {count}" for model, count in count_info.items()])

        ax.set_title(title, fontsize=14, weight='bold')
        ax.axis('off')

        plt.tight_layout()
        plt.show()

        # Print details
        print(f"\n{'='*60}")
        print(f"Scene {scene_idx} - Annotation Details")
        print(f"{'='*60}")
        for region_idx, region in enumerate(scene_data['regions']):
            points = np.array(region['points'])
            area = cv2.contourArea(points.astype(np.float32).reshape(-1, 1, 2))
            print(f"Region {region_idx+1}: {region['label']}, Area: {area:.0f} pixels")
        print(f"Total area: {sum(cv2.contourArea(np.array(r['points'], dtype=np.float32).reshape(-1, 1, 2)) for r in scene_data['regions']):.0f} pixels")


def plot_ground_truth_statistics(json_file_path):
    """
    Show overall annotation statistics.
    """
    with open(json_file_path, 'r') as f:
        labels = json.load(f)

    # Collect statistics
    total_books = 0
    model_counts = {}
    area_per_scene = {}

    for scene_name, scene_data in labels.items():
        scene_idx = int(scene_name.split('_')[1].split('.')[0])

        # Count books per model
        for model, count in scene_data['count'].items():
            if model not in model_counts:
                model_counts[model] = 0
            model_counts[model] += count
            total_books += count

        # Calculate total area per scene
        total_area = sum(
            cv2.contourArea(np.array(r['points'], dtype=np.float32).reshape(-1, 1, 2))
            for r in scene_data['regions']
        )
        area_per_scene[scene_idx] = total_area

    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # 1. Model distribution
    ax = axes[0, 0]
    models = list(model_counts.keys())
    counts = list(model_counts.values())
    ax.bar(range(len(models)), counts, color='skyblue')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=45, ha='right')
    ax.set_ylabel('Number of instances')
    ax.set_title('Model Distribution in Dataset')
    ax.grid(axis='y', alpha=0.3)

    # 2. Area per scene
    ax = axes[0, 1]
    scenes = sorted(area_per_scene.keys())
    areas = [area_per_scene[s] for s in scenes]
    ax.plot(scenes, areas, 'o-', color='coral', linewidth=2, markersize=8)
    ax.set_xlabel('Scene Index')
    ax.set_ylabel('Total Area (pixels)')
    ax.set_title('Total Annotated Area per Scene')
    ax.grid(True, alpha=0.3)

    # 3. Area histogram
    ax = axes[1, 0]
    ax.hist(areas, bins=15, color='lightgreen', edgecolor='black')
    ax.set_xlabel('Area (pixels)')
    ax.set_ylabel('Number of scenes')
    ax.set_title('Area Distribution per Scene')
    ax.grid(axis='y', alpha=0.3)

    # 4. Text statistics
    ax = axes[1, 1]
    ax.axis('off')
    stats_text = f"""
    DATASET STATISTICS

    Total scenes: {len(labels)}
    Total books: {total_books}
    Unique models: {len(model_counts)}

    Average area per scene: {np.mean(areas):.0f} pixels
    Min area: {np.min(areas):.0f} pixels
    Max area: {np.max(areas):.0f} pixels

    Most frequent models:
    """

    # Top 5 models
    sorted_models = sorted(model_counts.items(), key=lambda x: x[1], reverse=True)
    for i, (model, count) in enumerate(sorted_models[:5], 1):
        stats_text += f"\n  {i}. {model}: {count} instances"

    ax.text(0.1, 0.5, stats_text, fontsize=12, verticalalignment='center',
           family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

# 1. Visualize specific scenes
print("Visualizing selected scenes...")
visualize_ground_truth_annotations(
    '/content/drive/MyDrive/AssignmentsIPCV/labels.json',
    scenes_images,
    scene_indices=[1, 10, 15, 18]
)

# 2. Show overall statistics
print("\nOverall dataset statistics...")
plot_ground_truth_statistics('/content/drive/MyDrive/AssignmentsIPCV/labels.json')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
def evaluate_hyperparameters_with_gt(
    scenes_images,
    models_images,
    ground_truth_areas,
    hyperparams,
    verbose=False
):
    """
    Evaluate hyperparameters using ground truth areas.

    Returns:
        dict: Results containing error metrics
    """


    matches = instance_detect(scenes_images, models_images, hyperparams, DEBUG=False)

    scene_errors = []
    total_absolute_error = 0
    total_percentage_error = 0
    scenes_evaluated = 0

    for scene_idx, gt_area in ground_truth_areas.items():

        scene_matches = matches.get(scene_idx, [])
        area_info = calculate_total_area(scene_matches)
        predicted_area = area_info['total_area']

        metrics = compute_area_metrics(predicted_area, gt_area)
        scene_errors.append({
            'scene_idx': scene_idx,
            'predicted': predicted_area,
            'ground_truth': gt_area,
            'metrics': metrics
        })

        total_absolute_error += metrics['absolute_error']
        total_percentage_error += metrics['percentage_error']
        scenes_evaluated += 1

        if verbose:
            print(f"Scene {scene_idx}: Pred={predicted_area:.0f}, GT={gt_area:.0f}, "
                  f"Error={metrics['percentage_error']:.2f}%")

    mean_absolute_error = total_absolute_error / scenes_evaluated if scenes_evaluated > 0 else float('inf')
    mean_percentage_error = total_percentage_error / scenes_evaluated if scenes_evaluated > 0 else float('inf')

    total_detections = sum(len(m) for m in matches.values())
    total_predicted_area = sum(
        calculate_total_area(m)['total_area'] for m in matches.values()
    )

    return {
        'hyperparams': hyperparams,
        'mean_absolute_error': mean_absolute_error,
        'mean_percentage_error': mean_percentage_error,
        'total_detections': total_detections,
        'total_predicted_area': total_predicted_area,
        'scenes_evaluated': scenes_evaluated,
        'scene_errors': scene_errors,
        'matches': matches
    }


def grid_search_with_ground_truth(
    scenes_images,
    models_images,
    ground_truth_areas,
    param_grid,
    checkpoint_file='grid_search_checkpoint.json',
    save_every=10,
    resume=True,
    verbose=True
):
  """
  Performs a grid search using ground truth areas as the evaluation metric, with checkpoint support.

  Args:
      scenes_images: List of scene images
      models_images: List of model images
      ground_truth_areas: Dict {scene_idx: ground_truth_area}
      param_grid: Dict with lists of values for each parameter
      checkpoint_file: Name of the checkpoint file
      save_every: Save checkpoint every N combinations
      resume: If True, resume from existing checkpoint
      verbose: If True, print progress

  Returns:
      List of results sorted by error (best first)
  """

  # generate combination
  param_names = list(param_grid.keys())
  param_values = list(param_grid.values())
  all_combinations = list(itertools.product(*param_values))

  timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
  results_file = f"grid_search_results_{timestamp}.json"

  # load checkpoint
  results = []
  completed_combinations = set()

  if resume:
      loaded_results, loaded_completed, loaded_param_grid = load_checkpoint(checkpoint_file)

      if loaded_results is not None:
          if loaded_param_grid == param_grid:
              results = loaded_results
              completed_combinations = loaded_completed
              print(f"Resuming from checkpoint: {len(completed_combinations)} combinations already completed")
          else:
              print("Warning: Checkpoint param_grid differs from current param_grid")
              print("Starting fresh grid search...")
              completed_combinations = set()
              results = []

  if verbose:
      print(f"\n{'='*80}")
      print(f"GRID SEARCH WITH GROUND TRUTH")
      print(f"{'='*80}")
      print(f"Total parameter combinations: {len(all_combinations)}")
      print(f"Already completed: {len(completed_combinations)}")
      print(f"Remaining: {len(all_combinations) - len(completed_combinations)}")
      print(f"Scenes with ground truth: {len(ground_truth_areas)}")
      print(f"Total scenes to process: {len(scenes_images)}")
      print(f"Checkpoint file: {checkpoint_file}")
      print(f"Final results file: {results_file}")
      print(f"{'='*80}\n")

  remaining_combinations = [
      (idx, combo) for idx, combo in enumerate(all_combinations)
      if idx not in completed_combinations
  ]

  try:
      for combo_idx, combination in tqdm(remaining_combinations,
                                        desc="Grid Search Progress",
                                        initial=len(completed_combinations),
                                        total=len(all_combinations)):

          hyperparams = dict(zip(param_names, combination))

          if verbose and (len(results) + 1) % 10 == 0:
              print(f"\nTesting combination {len(results) + 1}/{len(all_combinations)}")
              print(f"Parameters: {hyperparams}")

          # evaluate combination
          result = evaluate_hyperparameters_with_gt(
              scenes_images,
              models_images,
              ground_truth_areas,
              hyperparams,
              verbose=False
          )

          result['combo_idx'] = combo_idx
          results.append(result)
          completed_combinations.add(combo_idx)

          if verbose and (len(results)) % 10 == 0:
              print(f"  Mean Absolute Error: {result['mean_absolute_error']:.0f} pixels")
              print(f"  Mean Percentage Error: {result['mean_percentage_error']:.2f}%")
              print(f"  Total Detections: {result['total_detections']}")

          # Save
          if len(results) % save_every == 0:
              save_checkpoint(results, completed_combinations, param_grid, checkpoint_file)
              if verbose:
                  print(f"\n   Checkpoint saved ({len(completed_combinations)}/{len(all_combinations)})")

  except KeyboardInterrupt:
      print("\n\nGrid search interrupted by user!")
      print(f"Saving checkpoint with {len(completed_combinations)} completed combinations...")
      save_checkpoint(results, completed_combinations, param_grid, checkpoint_file)
      print(f"Checkpoint saved to: {checkpoint_file}")
      print("\nYou can resume the grid search by running the script again.")
      raise

  except Exception as e:
      print(f"\n\nError during grid search: {e}")
      print(f"Saving checkpoint with {len(completed_combinations)} completed combinations...")
      save_checkpoint(results, completed_combinations, param_grid, checkpoint_file)
      print(f"Checkpoint saved to: {checkpoint_file}")
      raise

  results.sort(key=lambda x: x['mean_percentage_error'])

  save_grid_search_results(results, results_file)

  if os.path.exists(checkpoint_file):
      os.remove(checkpoint_file)
      print(f"\nCheckpoint file removed (grid search completed)")

  return results

In [20]:
def print_evaluation_results(json_file_path):
    """
    Print evaluation results from JSON file in a formatted way.

    Args:
        json_file_path: Path to the JSON results file
    """
    # Load JSON file
    with open(json_file_path, 'r') as f:
        data = json.load(f)

    best_config = data['best_configuration']

    # Print header
    print("=" * 80)
    print(f"{'BEST CONFIGURATION RESULTS':^80}")
    print("=" * 80)
    print()

    # Print rank
    print(f"Rank: {best_config['rank']}")
    print()

    # Print hyperparameters
    print("-" * 80)
    print("HYPERPARAMETERS")
    print("-" * 80)
    hyperparams = best_config['hyperparameters']
    for param, value in hyperparams.items():
        print(f"  {param:<25}: {value}")
    print()

    # Print overall metrics
    print("-" * 80)
    print("OVERALL METRICS")
    print("-" * 80)
    metrics = best_config['metrics']
    print(f"  {'Mean Absolute Error':<30}: {metrics['mean_absolute_error']:.2f}")
    print(f"  {'Mean Percentage Error':<30}: {metrics['mean_percentage_error']:.2f}%")
    print(f"  {'Total Detections':<30}: {metrics['total_detections']}")
    print(f"  {'Total Predicted Area':<30}: {metrics['total_predicted_area']:.2f}")
    print(f"  {'Scenes Evaluated':<30}: {metrics['scenes_evaluated']}")
    print()

    # Print per-scene errors
    print("-" * 80)
    print("PER-SCENE ERRORS")
    print("-" * 80)
    print(f"{'Scene':<8}{'Predicted':<15}{'Ground Truth':<15}{'Abs Error':<15}{'% Error':<12}{'Rel Error':<12}")
    print("-" * 80)

    for scene in best_config['per_scene_errors']:
        scene_idx = scene['scene_idx']
        predicted = scene['predicted_area']
        gt = scene['ground_truth_area']
        abs_err = scene['absolute_error']
        pct_err = scene['percentage_error']
        rel_err = scene['relative_error']

        print(f"{scene_idx:<8}{predicted:<15.2f}{gt:<15.2f}{abs_err:<15.2f}{pct_err:<12.2f}{rel_err:<12.3f}")

    print("=" * 80)
    print()

    # Print summary statistics
    print("-" * 80)
    print("SUMMARY STATISTICS")
    print("-" * 80)

    per_scene = best_config['per_scene_errors']
    errors = [s['absolute_error'] for s in per_scene]
    pct_errors = [s['percentage_error'] for s in per_scene]

    print(f"  {'Number of scenes':<30}: {len(per_scene)}")
    print(f"  {'Min absolute error':<30}: {min(errors):.2f}")
    print(f"  {'Max absolute error':<30}: {max(errors):.2f}")
    print(f"  {'Min percentage error':<30}: {min(pct_errors):.2f}%")
    print(f"  {'Max percentage error':<30}: {max(pct_errors):.2f}%")

    # Count scenes with zero predictions
    zero_pred = sum(1 for s in per_scene if s['predicted_area'] == 0)
    if zero_pred > 0:
        print(f"  {'Scenes with zero predictions':<30}: {zero_pred}")

    print("=" * 80)

print_evaluation_results('/content/drive/MyDrive/AssignmentsIPCV/results.json')

                           BEST CONFIGURATION RESULTS                           

Rank: 1

--------------------------------------------------------------------------------
HYPERPARAMETERS
--------------------------------------------------------------------------------
  nfeatures                : 0
  nOctaveLayers            : 3
  contrastThreshold        : 0.015
  edgeThreshold            : 15
  sigma                    : 1.6
  k                        : 2
  ratio_test               : 0.85
  ransac_threshold         : 10.0
  min_matches              : 3
  blur_kernel              : 3
  use_clahe                : False
  use_hist_eq              : False

--------------------------------------------------------------------------------
OVERALL METRICS
--------------------------------------------------------------------------------
  Mean Absolute Error           : 12931.77
  Mean Percentage Error         : 32.48%
  Total Detections              : 44
  Total Predicted Area          : 7782

## Main Loop

**Note**: By setting `DEBUG = True`, you can visualize the detection and matching process to better understand and debug failing instances (this produces a lot of output, so use it only on specific scenes or models).


In [6]:
def preprocess_for_sift(gray):
    blurred = cv2.GaussianBlur(gray, (13, 13), 0)
    equalized = cv2.equalizeHist(blurred)
    return equalized


DEBUG = False
all_matches = []
for SCENE_IDX in tqdm(range(len(scenes_images)), desc="Processing scenes"):
    scenes_image = scenes_images[SCENE_IDX].copy()

    matches = []
    for idx, m in enumerate(models_images):
        model_image = m.copy()

        for i in range(10):
            scene_corners = instance_detect(
                scenes_image,
                model_image,
                preprocess=None,
                plot_matches=DEBUG,
                plot_rectangle=DEBUG,
                plot_descriptors=DEBUG,
                debug=DEBUG,
            )

            is_valid, reason = is_rectangle_valid(scene_corners, scenes_image.shape)
            if DEBUG:
                print(
                    f"Model {idx}, Iteration {i}, Valid: {is_valid}, Reason: {reason}"
                )
            if not is_valid:
                break
            # Mask the detected area in the scene image to find another instance
            mask = np.zeros(scenes_image.shape[:2], dtype=np.uint8)

            matches.append((model_image, scene_corners, idx, SCENE_IDX))
            all_matches.append((model_image, scene_corners, idx, SCENE_IDX))
            cv2.fillPoly(mask, [np.int32(scene_corners)], (255))
            scenes_image = cv2.bitwise_and(
                scenes_image, scenes_image, mask=cv2.bitwise_not(mask)
            )
    plot_detections(scenes_images, matches, SCENE_IDX=SCENE_IDX)

Output hidden; open in https://colab.research.google.com to view.

## Output in the required format


In [9]:
for scene_idx in range(len(scenes_images)):
    print(f"\nDetections in scene {scene_idx}:")
    scene_matches = [m for m in all_matches if m[3] == scene_idx]
    for model_idx in range(len(models_images)):
        model_detections = [m for m in scene_matches if m[2] == model_idx]
        if len(model_detections) > 0:
            print(f"    Book {model_idx}: {len(model_detections)} instance(s) found:")
            for det_idx, det in enumerate(model_detections):
                coords = det[1].astype(int)
                print(
                    f"        Instance {det_idx}: 'top_left': {coords[0][0].tolist()}, 'bottom_right': {coords[1][0].tolist()}, 'bottom_left': {coords[2][0].tolist()}, 'top_right': {coords[3][0].tolist()}, 'area': {int(cv2.contourArea(coords))}px"
                )


Detections in scene 0:

Detections in scene 1:
    Book 18: 2 instance(s) found:
        Instance 0: 'top_left': [441, 42], 'bottom_right': [490, 42], 'bottom_left': [490, 521], 'top_right': [441, 521], 'area': 23471px
        Instance 1: 'top_left': [489, 38], 'bottom_right': [537, 39], 'bottom_left': [530, 514], 'top_right': [481, 513], 'area': 23045px

Detections in scene 2:
    Book 17: 1 instance(s) found:
        Instance 0: 'top_left': [283, 25], 'bottom_right': [318, 25], 'bottom_left': [318, 503], 'top_right': [283, 503], 'area': 16730px

Detections in scene 3:
    Book 16: 2 instance(s) found:
        Instance 0: 'top_left': [377, 209], 'bottom_right': [426, 209], 'bottom_left': [426, 542], 'top_right': [377, 542], 'area': 16317px
        Instance 1: 'top_left': [425, 209], 'bottom_right': [475, 210], 'bottom_left': [467, 549], 'top_right': [417, 548], 'area': 16958px

Detections in scene 4:
    Book 14: 2 instance(s) found:
        Instance 0: 'top_left': [92, 1], 'bottom_r

# Comments

The proposed solution based on SIFT descriptors can find most of the books in the scenes, even though it struggles with some of the most difficult ones (e.g., scene27 or scene16).
However, the proposed approach relies on many hyperparameters, that have been tuned empirically to achieve the best results on the provided dataset. We tried to implement a Grid Search but unfortunatly, it did not The main hyperparameters are:

- Detection hyperparameters:
  - SIFT parameters (e.g., number of features, contrast threshold, edge threshold, number of octaves, etc.);
- Filtering hyperparameters:
  - Lowe's ratio threshold;
  - Minimum number of inliers to accept a detection;
  - RANSAC reprojection threshold;
  - Rectangle validity criteria (e.g., area range, rectangularity extent, etc.);

Of course, the dependence on hyperparameters can lead to overfitting on the provided dataset, and the performance of the proposed approach may degrade on different datasets or real-world scenarios.
The hyperparameters search was not easy, because chaging one hyperparameter to fix a certain model-scene pair could lead to worse performance on other pairs.


## Evaluation criteria

1. **Clarity and conciseness**. Present your work in a readable way: format your code and comment every important step;

2. **Procedural correctness**. There are several ways to solve the assignment. Design your own sound approach and justify every decision you make;

3. **Correctness of results**. Try to solve as many instances as possible. You should be able to solve all the instances of the assignment, however, a thoroughly justified and sound procedure with a lower number of solved instances will be valued **more** than a poorly designed and justified approach that solves more or all instances.
